# 1 — Data collection and exploration

**AIG230 Final Project · Group 7**

The corpus is rebuilt from the official Tatoeba exports rather than downloaded
as a ready-made bitext, so that we control the snapshot and keep the sentence
ids for traceability.

If `artifacts/results/dataset_stats.json` does not exist yet, run:

```bash
python -m nmt.data.build
```


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from nmt.utils.io import project_root, read_json
from nmt.viz.style import use_style

use_style()
print("project root:", ROOT)

In [ ]:
stats = read_json(ROOT / "artifacts" / "results" / "dataset_stats.json")
pairing, cleaning, split = stats["pairing"], stats["cleaning"], stats["split"]

print(f"English sentences in Tatoeba : {pairing['english_sentences']:,}")
print(f"Spanish sentences            : {pairing['spanish_sentences']:,}")
print(f"Translation links scanned    : {pairing['links_scanned']:,}")
print(f"EN-ES pairs recovered        : {pairing['pairs_found']:,}")
print(f"Pairs after cleaning         : {cleaning['kept_pairs']:,} "
      f"({cleaning['retention_rate']:.2%} retained)")

## Why so little is removed

Tatoeba is human-curated, so the aggressive cleaning that web-crawled corpora
need is unnecessary here. The filters earn their place by *confirming* data
quality and by catching the handful of genuinely broken rows.

In [ ]:
import pandas as pd

removed = pd.Series(cleaning["removed"], name="pairs removed").sort_values(ascending=False)
display(removed.to_frame())

for reason, examples in list(cleaning["examples_removed"].items())[:4]:
    print(f"\n--- {reason} ---")
    for en, es in examples[:2]:
        print(f"  EN: {en}\n  ES: {es}")

## Splits, and the leakage that a naive split would cause

Tatoeba is a translation **graph**: one English sentence links to several
Spanish ones. Splitting the pair list at random puts paraphrases of the same
sentence on both sides of the train/test boundary. We split connected
components instead.

In [ ]:
print("split sizes      :", split["split_pairs"])
print("components       :", f"{split['components']:,}")
print("largest component:", split["largest_component_pairs"], "pairs")
print("singletons       :", f"{split['singleton_components']:,}")
print()
print("Leakage check (sentences shared between splits):")
for key, value in split["leakage_check"].items():
    print(f"  {key:32s} {value}")

In [ ]:
# Demonstrate the failure the component split prevents.
import random
from nmt.data.build import read_split

train_pairs = read_split(ROOT / "data" / "processed" / "train.tsv")

from collections import Counter
english_counts = Counter(en for en, _ in train_pairs)
multi = [s for s, n in english_counts.most_common(5)]
print("English sentences with the most alternative Spanish translations:\n")
for sentence in multi[:3]:
    print(f"  {sentence!r}  ->  {english_counts[sentence]} translations")
    for en, es in train_pairs:
        if en == sentence:
            print(f"        {es}")
    print()

## Length distributions

In [ ]:
from nmt.viz.data_plots import plot_length_distributions

plot_length_distributions(ROOT / "data" / "processed", ROOT / "reports" / "figures" / "data_lengths")
from IPython.display import Image
Image(str(ROOT / "reports" / "figures" / "data_lengths.png"), width=900)

## Vocabulary: the case for subword tokenisation

Spanish marks person, number, tense and mood on the verb, agrees adjectives and
articles for gender and number, and attaches clitic pronouns to infinitives.
English does almost none of this — so Spanish contributes far more distinct
surface forms for the same amount of running text.

In [ ]:
vocab = stats["splits"]["train"]["vocabulary"]
summary = pd.DataFrame({
    lang: {
        "types": v["types"],
        "tokens": v["tokens"],
        "type/token ratio": round(v["type_token_ratio"], 4),
        "hapax %": round(100 * v["hapax_fraction"], 1),
        "coverage @10k": round(100 * v["coverage_top_10000"], 2),
    }
    for lang, v in vocab.items()
}).T
display(summary)

print(f"\nSpanish has {vocab['es']['types'] / vocab['en']['types']:.2f}x "
      "as many word types as English.")

In [ ]:
for name in ("validation", "test"):
    oov = stats["splits"][name]["word_oov"]
    print(f"{name:11s} OOV rate — EN {oov['en']['unk_rate']:.2%}   "
          f"ES {oov['es']['unk_rate']:.2%}")

print("\nMost frequent Spanish words missing from the 32k word vocabulary:")
print([w for w, _ in stats["splits"]["test"]["word_oov"]["es"]["most_common_unk"][:15]])

In [ ]:
from nmt.viz.data_plots import plot_vocabulary_growth, plot_subword_effect

plot_vocabulary_growth(stats, ROOT / "reports" / "figures" / "data_vocabulary")
plot_subword_effect(stats, ROOT / "reports" / "figures" / "data_subwords")
Image(str(ROOT / "reports" / "figures" / "data_vocabulary.png"), width=900)

## What this means for the model

| Observation | Consequence |
|---|---|
| Sentences are short (median ~6 words) | `max_length = 64` discards a negligible tail; batches can be capped by token count |
| Spanish has ~1.6x the types of English | subword vocabulary; report chrF2 alongside BLEU |
| ~40% of types are hapax | a modest 16k vocabulary, so rare entries are still updated often enough to learn |
| Many-to-many links | component-aware splitting, verified on every build |
